# Algorithm Analysis and Simulation Toolkit
**Final Project — Algorithms & Complexity | Term 2, SY 2025–2026**

---
### Contents
1. [Part 1 — Sorting Algorithm Comparison](#part1)
2. [Part 2 — MST: Kruskal's & Prim's](#part2)
3. [Part 3 — Recursive Function Simulation](#part3)
4. [Summary](#summary)

In [ ]:
# =============================================================================
# IMPORTS — only what is actually used in this notebook
# =============================================================================
import time
import random
import heapq
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from collections import defaultdict
from IPython.display import display
import pandas as pd

# Fix the random seed so benchmark results are reproducible across runs
random.seed(42)

# Consistent plot styling
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'monospace'

print('All imports loaded successfully.')

---
<a id='part1'></a>
## Part 1 — Sorting Algorithm Comparison

All 8 algorithms are implemented so that each returns `(sorted_list, comparisons, swaps)`,
enabling a fair side-by-side comparison on identical datasets.

**Complexity reference**

| Algorithm | Best | Average | Worst | Space | Stable? |
|---|---|---|---|---|---|
| Bubble Sort | O(n) | O(n²) | O(n²) | O(1) | Yes |
| Selection Sort | O(n²) | O(n²) | O(n²) | O(1) | No |
| Insertion Sort | O(n) | O(n²) | O(n²) | O(1) | Yes |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) | Yes |
| Quick Sort | O(n log n) | O(n log n) | O(n²) | O(log n) | No |
| Randomized Quick Sort | O(n log n) | O(n log n) | O(n²)* | O(log n) | No |
| Counting Sort | O(n + k) | O(n + k) | O(n + k) | O(k) | Yes |
| Radix Sort (LSD) | O(d·n) | O(d·n) | O(d·n) | O(n + b) | Yes |

\* Randomized Quick Sort worst-case O(n²) occurs with negligible probability — expected O(n log n).

> **Note on the 'Operations' metric:** Bubble/Selection/Insertion/Merge/Quick sorts track
> traditional key-comparison counts. Counting Sort and Radix Sort are non-comparison sorts;
> for those the 'Operations' column counts element read/write passes per digit/bucket stage,
> not key comparisons.

In [ ]:
# =============================================================================
# SORTING ALGORITHMS
# Each function accepts a list and returns (sorted_list, operations, swaps).
# operations = key-comparison count for comparison sorts;
#              element-pass count for non-comparison sorts (Counting, Radix).
# swaps      = in-place swap count (0 for out-of-place sorts).
# =============================================================================

# ── BUBBLE SORT ──────────────────────────────────────────────────────────────
def bubble_sort(arr):
    """Optimized bubble sort with early-exit when no swaps occur in a pass.
    Time: O(n) best, O(n²) average/worst. Space: O(1). Stable."""
    a = arr[:]
    n = len(a)
    num_comparisons = num_swaps = 0
    for pass_num in range(n):
        swapped = False
        for j in range(0, n - pass_num - 1):
            num_comparisons += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                num_swaps += 1
                swapped = True
        if not swapped:   # already sorted — early exit
            break
    return a, num_comparisons, num_swaps


# ── SELECTION SORT ───────────────────────────────────────────────────────────
def selection_sort(arr):
    """Selection sort: finds the minimum and places it in position each pass.
    Time: O(n²) all cases. Space: O(1). Not stable."""
    a = arr[:]
    n = len(a)
    num_comparisons = num_swaps = 0
    for i in range(n):
        min_idx = i
        for j in range(i + 1, n):
            num_comparisons += 1
            if a[j] < a[min_idx]:
                min_idx = j
        if min_idx != i:
            a[i], a[min_idx] = a[min_idx], a[i]
            num_swaps += 1
    return a, num_comparisons, num_swaps


# ── INSERTION SORT ───────────────────────────────────────────────────────────
def insertion_sort(arr):
    """Insertion sort: builds sorted prefix by inserting each element in place.
    Time: O(n) best, O(n²) average/worst. Space: O(1). Stable."""
    a = arr[:]
    num_comparisons = num_swaps = 0
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0:
            num_comparisons += 1
            if a[j] > key:
                a[j + 1] = a[j]   # shift right
                num_swaps += 1
                j -= 1
            else:
                break
        a[j + 1] = key
    return a, num_comparisons, num_swaps


# ── MERGE SORT ───────────────────────────────────────────────────────────────
def merge_sort(arr):
    """Top-down merge sort. Out-of-place — returns a fresh sorted list.
    Time: O(n log n) all cases. Space: O(n). Stable.
    Swaps are always 0 because merging builds a new list rather than swapping in-place."""
    num_comparisons_ref = [0]

    def _merge(left, right):
        """Merge two sorted lists into one, counting key comparisons."""
        merged = []
        i = j = 0
        while i < len(left) and j < len(right):
            num_comparisons_ref[0] += 1
            if left[i] <= right[j]:
                merged.append(left[i])
                i += 1
            else:
                merged.append(right[j])
                j += 1
        merged.extend(left[i:])
        merged.extend(right[j:])
        return merged

    def _sort(a):
        if len(a) <= 1:
            return a
        mid = len(a) // 2
        return _merge(_sort(a[:mid]), _sort(a[mid:]))

    sorted_arr = _sort(arr[:])
    return sorted_arr, num_comparisons_ref[0], 0   # 0 swaps (out-of-place)


# ── QUICK SORT (deterministic — last-element pivot) ───────────────────────────
def quick_sort(arr):
    """Lomuto partition scheme, pivot = last element.
    Time: O(n log n) average, O(n²) worst (sorted input). Space: O(log n) stack. Not stable."""
    if not arr:
        return [], 0, 0
    num_comparisons_ref = [0]
    num_swaps_ref = [0]

    def _partition(a, lo, hi):
        pivot = a[hi]
        i = lo - 1
        for j in range(lo, hi):
            num_comparisons_ref[0] += 1
            if a[j] <= pivot:
                i += 1
                a[i], a[j] = a[j], a[i]
                num_swaps_ref[0] += 1
        a[i + 1], a[hi] = a[hi], a[i + 1]   # place pivot in final position
        num_swaps_ref[0] += 1
        return i + 1

    def _sort(a, lo, hi):
        if lo < hi:
            pivot_idx = _partition(a, lo, hi)
            _sort(a, lo, pivot_idx - 1)
            _sort(a, pivot_idx + 1, hi)

    a = arr[:]
    _sort(a, 0, len(a) - 1)
    return a, num_comparisons_ref[0], num_swaps_ref[0]


# ── RANDOMIZED QUICK SORT ─────────────────────────────────────────────────────
def random_quick_sort(arr):
    """Quick sort with random pivot selection to avoid worst-case on sorted input.
    Time: O(n log n) expected, O(n²) with negligible probability. Space: O(log n). Not stable."""
    if not arr:
        return [], 0, 0
    num_comparisons_ref = [0]
    num_swaps_ref = [0]

    def _partition(a, lo, hi):
        # Swap a random element into the pivot position before partitioning
        rand_idx = random.randint(lo, hi)
        a[rand_idx], a[hi] = a[hi], a[rand_idx]
        num_swaps_ref[0] += 1
        pivot = a[hi]
        i = lo - 1
        for j in range(lo, hi):
            num_comparisons_ref[0] += 1
            if a[j] <= pivot:
                i += 1
                a[i], a[j] = a[j], a[i]
                num_swaps_ref[0] += 1
        a[i + 1], a[hi] = a[hi], a[i + 1]   # place pivot in final position
        num_swaps_ref[0] += 1
        return i + 1

    def _sort(a, lo, hi):
        if lo < hi:
            pivot_idx = _partition(a, lo, hi)
            _sort(a, lo, pivot_idx - 1)
            _sort(a, pivot_idx + 1, hi)

    a = arr[:]
    _sort(a, 0, len(a) - 1)
    return a, num_comparisons_ref[0], num_swaps_ref[0]


# ── COUNTING SORT ─────────────────────────────────────────────────────────────
def counting_sort(arr):
    """Integer sort for non-negative values. Not a comparison sort.
    Time: O(n + k) where k = max value. Space: O(k). Stable.
    'Operations' counts element read passes, not key comparisons."""
    if not arr:
        return [], 0, 0
    a = arr[:]
    max_val = max(a)
    count_buckets = [0] * (max_val + 1)
    num_operations = 0
    for val in a:
        count_buckets[val] += 1
        num_operations += 1   # one element read per pass
    result = []
    for val, freq in enumerate(count_buckets):
        result.extend([val] * freq)
    return result, num_operations, 0   # 0 swaps — output built from scratch


# ── RADIX SORT (LSD, base-10) ─────────────────────────────────────────────────
def radix_sort(arr):
    """Least-Significant-Digit radix sort for non-negative integers. Not a comparison sort.
    Time: O(d * n) where d = number of digits in max value. Space: O(n + 10). Stable.
    'Operations' counts element passes per digit stage, not key comparisons."""
    if not arr:
        return [], 0, 0
    a = arr[:]
    num_operations = 0
    max_val = max(a)
    digit_place = 1
    while max_val // digit_place > 0:
        digit_buckets = [[] for _ in range(10)]
        for num in a:
            digit = (num // digit_place) % 10
            digit_buckets[digit].append(num)
            num_operations += 1
        a = [num for bucket in digit_buckets for num in bucket]
        digit_place *= 10
    return a, num_operations, 0   # 0 swaps — buckets rebuilt each pass


print('All 8 sorting algorithms defined.')

In [ ]:
# =============================================================================
# BENCHMARK RUNNER
# =============================================================================

ALGORITHMS = {
    'Bubble Sort':        bubble_sort,
    'Selection Sort':     selection_sort,
    'Insertion Sort':     insertion_sort,
    'Merge Sort':         merge_sort,
    'Quick Sort':         quick_sort,
    'Random-Quick Sort':  random_quick_sort,
    'Counting Sort':      counting_sort,
    'Radix Sort':         radix_sort,
}


def benchmark(dataset_size=1000, value_range=(0, 9999), runs=5):
    """Run every algorithm on the same random dataset and collect performance metrics.

    Timing, operation counts, and swap counts are all averaged over `runs` trials
    to reduce noise. All timing uses time.perf_counter() for high resolution.

    Args:
        dataset_size: Number of elements in the test array.
        value_range:  (min, max) integer range for generated values.
        runs:         Number of timing trials to average.

    Returns:
        DataFrame ranked by ascending average time.
    """
    dataset = [random.randint(*value_range) for _ in range(dataset_size)]
    results = []

    for alg_name, sort_fn in ALGORITHMS.items():
        trial_times  = []
        total_ops    = 0
        total_swaps  = 0

        for _ in range(runs):
            start = time.perf_counter()
            _, num_ops, num_swaps = sort_fn(dataset)
            elapsed_ms = (time.perf_counter() - start) * 1000
            trial_times.append(elapsed_ms)
            total_ops   += num_ops
            total_swaps += num_swaps

        results.append({
            'Algorithm':   alg_name,
            'Time (ms)':   round(sum(trial_times) / runs, 4),
            'Operations':  total_ops   // runs,   # averaged over runs
            'Swaps':       total_swaps // runs,   # averaged over runs
        })

    df = pd.DataFrame(results).sort_values('Time (ms)').reset_index(drop=True)
    df.index += 1   # rank starts at 1
    print(f'Dataset Size: {dataset_size}  |  Value Range: {value_range}  |  Trials averaged: {runs}\n')
    display(df)
    return df


results_df = benchmark(dataset_size=1000)

In [ ]:
# =============================================================================
# PERFORMANCE CHARTS — Time, Operations, Swaps
# ALGO_COLORS is defined here and reused by all subsequent chart cells.
# =============================================================================

ALGO_COLORS = plt.cm.tab10.colors   # 10 distinct colors for 8 algorithms

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Sorting Algorithm Benchmark — n=1000', fontsize=13, fontweight='bold')

metrics = ['Time (ms)', 'Operations', 'Swaps']
for ax, metric in zip(axes, metrics):
    sorted_df = results_df.sort_values(metric, ascending=False)
    bars = ax.barh(
        sorted_df['Algorithm'],
        sorted_df[metric],
        color=ALGO_COLORS[:len(sorted_df)],
        edgecolor='white',
    )
    ax.set_xlabel(metric)
    ax.set_title(metric)
    fmt = '%.3f' if metric == 'Time (ms)' else '%d'
    ax.bar_label(bars, fmt=fmt, padding=3, fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('sorting_benchmark.png', bbox_inches='tight')
plt.show()
print('Chart saved to sorting_benchmark.png')

In [ ]:
# =============================================================================
# SCALABILITY TEST — Average Time vs. Dataset Size
# Each (algorithm, size) pair is timed over SCALE_RUNS trials and averaged
# to reduce measurement noise.
# =============================================================================

SIZES      = [100, 500, 1000, 2000, 5000]
SCALE_RUNS = 3

scale_times = {alg_name: [] for alg_name in ALGORITHMS}

for size in SIZES:
    dataset = [random.randint(0, 9999) for _ in range(size)]
    for alg_name, sort_fn in ALGORITHMS.items():
        trial_times = []
        for _ in range(SCALE_RUNS):
            start = time.perf_counter()
            sort_fn(dataset)
            trial_times.append((time.perf_counter() - start) * 1000)
        scale_times[alg_name].append(sum(trial_times) / SCALE_RUNS)

fig, ax = plt.subplots(figsize=(11, 6))
for i, (alg_name, times) in enumerate(scale_times.items()):
    ax.plot(SIZES, times, marker='o', label=alg_name,
            color=ALGO_COLORS[i], linewidth=2)

ax.set_xlabel('Dataset Size (n)', fontsize=11)
ax.set_ylabel('Average Time (ms)', fontsize=11)
ax.set_title('Algorithm Scalability — Average Time vs. Dataset Size', fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('sorting_scalability.png', bbox_inches='tight')
plt.show()
print('Chart saved to sorting_scalability.png')

In [ ]:
# =============================================================================
# STEP-BY-STEP TRACE — Bubble Sort on a small array
# Prints each swap so the progression of the algorithm is clearly visible.
# =============================================================================

def bubble_sort_trace(arr, max_steps=20):
    """Run bubble sort and print every swap, up to max_steps events.

    Args:
        arr:       Input list (not modified in place).
        max_steps: Maximum swap events to print before truncating output.

    Returns:
        The sorted list.
    """
    a = arr[:]
    n = len(a)
    step = 0
    print(f'Initial: {a}')
    for pass_num in range(n):
        for j in range(0, n - pass_num - 1):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                step += 1
                print(f'  Step {step:3d} | pass={pass_num + 1}, swap [{j}] & [{j + 1}]: {a}')
                if step >= max_steps:
                    print('  ... (trace truncated at max_steps)')
                    return a
    print(f'Sorted:  {a}')
    return a


sample = random.sample(range(1, 30), 10)
print('=== Bubble Sort Step-by-Step Trace ===')
bubble_sort_trace(sample)

In [ ]:
# =============================================================================
# BEST / AVERAGE / WORST CASE ANALYSIS
# Tests each algorithm on already-sorted, random, and reverse-sorted input.
# Each (algorithm, case) pair is timed over CASE_RUNS trials and averaged.
# =============================================================================

CASE_N    = 500
CASE_RUNS = 3

cases = {
    'Best (sorted)':    list(range(CASE_N)),
    'Average (random)': random.sample(range(CASE_N * 10), CASE_N),
    'Worst (reversed)': list(range(CASE_N, 0, -1)),
}

case_results = []
for case_label, data in cases.items():
    for alg_name, sort_fn in ALGORITHMS.items():
        trial_times = []
        for _ in range(CASE_RUNS):
            start = time.perf_counter()
            sort_fn(data)
            trial_times.append((time.perf_counter() - start) * 1000)
        avg_ms = round(sum(trial_times) / CASE_RUNS, 4)
        case_results.append({'Case': case_label, 'Algorithm': alg_name, 'Time (ms)': avg_ms})

case_df = pd.DataFrame(case_results)
pivot   = case_df.pivot(index='Algorithm', columns='Case', values='Time (ms)')
pivot   = pivot.sort_values('Average (random)')   # sort rows by average-case time

print(f'Best / Average / Worst Case — Mean Time (ms) over {CASE_RUNS} runs — n={CASE_N}\n')
display(pivot)

---
<a id='part2'></a>
## Part 2 — Minimum Spanning Tree: Kruskal's & Prim's

Both algorithms find the minimum spanning tree (MST) of a weighted undirected graph.
When all edge weights are distinct, the MST is unique and both algorithms produce the
same total cost.

| Algorithm | Strategy | Time Complexity | Data Structure |
|---|---|---|---|
| Kruskal's | Edge-greedy — sort all edges, add if no cycle formed | O(E log E) | Union-Find (DSU) |
| Prim's | Vertex-greedy — grow MST from a start vertex | O(E log V) | Min-heap |

Both algorithms are implemented with step-by-step verbose tracing and visualized
side-by-side using NetworkX.

In [ ]:
# =============================================================================
# MST UTILITIES: UnionFind, build_adj, kruskal, prim
# =============================================================================

# ── UNION-FIND (Disjoint Set Union) ──────────────────────────────────────────
class UnionFind:
    """Disjoint Set Union with path compression and union by rank.
    Both find() and union() run in nearly O(1) amortized time."""

    def __init__(self, n):
        """Initialize n singleton sets (0-indexed)."""
        self.parent = list(range(n))
        self.rank   = [0] * n

    def find(self, x):
        """Return the representative of x's set (with path compression)."""
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        """Merge the sets of x and y. Returns False if already in the same set."""
        root_x, root_y = self.find(x), self.find(y)
        if root_x == root_y:
            return False   # same set — this edge would form a cycle
        # Attach smaller-rank tree under the larger-rank root
        if self.rank[root_x] < self.rank[root_y]:
            root_x, root_y = root_y, root_x
        self.parent[root_y] = root_x
        if self.rank[root_x] == self.rank[root_y]:
            self.rank[root_x] += 1
        return True


# ── HELPER — build adjacency list from an edge list ──────────────────────────
def build_adj(edges):
    """Convert a list of (u, v, weight) tuples to an undirected adjacency dict.

    Returns:
        dict mapping each vertex to a list of (neighbor, weight) pairs.
    """
    adj = defaultdict(list)
    for u, v, w in edges:
        adj[u].append((v, w))
        adj[v].append((u, w))   # undirected: both directions added
    return dict(adj)


# ── KRUSKAL'S ALGORITHM ───────────────────────────────────────────────────────
def kruskal(vertices, edges, verbose=True):
    """Find the MST using Kruskal's algorithm (greedy edge selection).

    Strategy:
        1. Sort all edges by ascending weight.
        2. Add an edge to the MST if it does not form a cycle (Union-Find).
        3. Stop when |MST edges| == |vertices| - 1.

    Time: O(E log E) dominated by the initial edge sort.

    Args:
        vertices: List of vertex labels.
        edges:    List of (u, v, weight) tuples.
        verbose:  Print step-by-step trace if True.

    Returns:
        (mst_edges, total_cost)
    """
    n = len(vertices)
    vertex_index = {v: i for i, v in enumerate(vertices)}
    uf = UnionFind(n)
    mst_edges = []

    sorted_edges = sorted(edges, key=lambda e: e[2])

    if verbose:
        print("=== KRUSKAL'S ALGORITHM ===")
        print(f'Vertices : {vertices}')
        print('Edges sorted by weight:')
        for u, v, w in sorted_edges:
            print(f'  ({u}--{v}, w={w})')
        print()

    for step_num, (u, v, w) in enumerate(sorted_edges, start=1):
        ui, vi = vertex_index[u], vertex_index[v]
        if uf.union(ui, vi):
            mst_edges.append((u, v, w))
            action = 'ADDED'
        else:
            action = 'SKIPPED (would form cycle)'
        if verbose:
            print(f'Step {step_num:2d} | Edge ({u}--{v}, w={w}) -> {action}')
        if len(mst_edges) == n - 1:
            break

    total_cost = sum(w for _, _, w in mst_edges)
    if verbose:
        print(f'\nMST Edges:')
        for u, v, w in mst_edges:
            print(f'  {u} -- {v}  (weight {w})')
        print(f'Total MST Cost = {total_cost}')

    return mst_edges, total_cost


# ── PRIM'S ALGORITHM ─────────────────────────────────────────────────────────
def prim(vertices, adj, start=None, verbose=True):
    """Find the MST using Prim's algorithm (greedy vertex expansion).

    Strategy:
        Maintain a min-heap of crossing edges (one endpoint in the visited set,
        one outside). Greedily add the minimum-weight crossing edge and expand
        the visited set until all vertices are included.

    Time: O(E log V) with a binary min-heap.

    Args:
        vertices: List of vertex labels.
        adj:      Adjacency dict returned by build_adj().
        start:    Starting vertex label (defaults to vertices[0]).
        verbose:  Print step-by-step trace if True.

    Returns:
        (mst_edges, total_cost)
    """
    if start is None:
        start = vertices[0]

    visited  = set()
    mst_edges = []
    # Heap entries: (weight, from_vertex, to_vertex)
    # The sentinel entry for the start vertex uses weight=0, from_vertex=None
    min_heap = [(0, None, start)]

    if verbose:
        print("=== PRIM'S ALGORITHM ===")
        print(f'Vertices        : {vertices}')
        print(f'Starting vertex : {start}\n')

    step_num = 0
    while min_heap:
        weight, from_vertex, to_vertex = heapq.heappop(min_heap)
        if to_vertex in visited:
            continue   # stale heap entry
        visited.add(to_vertex)

        if from_vertex is not None:   # skip the sentinel start entry
            mst_edges.append((from_vertex, to_vertex, weight))
            step_num += 1
            if verbose:
                print(f'Step {step_num:2d} | Add edge ({from_vertex}--{to_vertex}, w={weight}) '
                      f'| MST so far: {[(a, b, c) for a, b, c in mst_edges]}')

        for neighbor, edge_weight in adj.get(to_vertex, []):
            if neighbor not in visited:
                heapq.heappush(min_heap, (edge_weight, to_vertex, neighbor))

    total_cost = sum(w for _, _, w in mst_edges)
    if verbose:
        print(f'\nMST Edges:')
        for u, v, w in mst_edges:
            print(f'  {u} -- {v}  (weight {w})')
        print(f'Total MST Cost = {total_cost}')

    return mst_edges, total_cost


print('MST utilities defined: UnionFind, build_adj, kruskal, prim')

In [ ]:
# =============================================================================
# SAMPLE GRAPH — classic 6-vertex weighted undirected graph
# =============================================================================

VERTICES = [1, 2, 3, 4, 5, 6]
EDGES = [
    (1, 2, 4), (1, 3, 3),
    (2, 3, 5), (2, 4, 6),
    (3, 4, 7), (3, 5, 8),
    (4, 5, 9), (4, 6, 5),
    (5, 6, 6),
]

ADJ = build_adj(EDGES)

k_mst, k_cost = kruskal(VERTICES, EDGES, verbose=True)

In [ ]:
# Run Prim's on the same graph starting from vertex 1
p_mst, p_cost = prim(VERTICES, ADJ, start=1, verbose=True)

In [ ]:
# =============================================================================
# GRAPH VISUALIZATION — Full Graph, Kruskal's MST, Prim's MST side-by-side
# =============================================================================

def draw_graph(ax, title, vertices, edges, mst_edges=None):
    """Draw a weighted undirected graph on ax, highlighting MST edges in red.

    Args:
        ax:        Matplotlib Axes to draw on.
        title:     Plot title string.
        vertices:  List of vertex labels.
        edges:     All graph edges as (u, v, weight) tuples.
        mst_edges: Subset of edges forming the MST (highlighted in red).
                   None means draw all edges in gray (full-graph view).
    """
    G = nx.Graph()
    G.add_nodes_from(vertices)
    for u, v, w in edges:
        G.add_edge(u, v, weight=w)

    pos = nx.spring_layout(G, seed=42)

    mst_set = set()
    if mst_edges:
        for u, v, _ in mst_edges:
            mst_set.add((u, v))
            mst_set.add((v, u))   # undirected lookup

    edge_colors = ['#e74c3c' if (u, v) in mst_set else '#bdc3c7' for u, v in G.edges()]
    edge_widths = [3.5      if (u, v) in mst_set else 1.0        for u, v in G.edges()]

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color='#2c3e50', node_size=600)
    nx.draw_networkx_labels(G, pos, ax=ax, font_color='white', font_weight='bold')
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_colors, width=edge_widths)
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax, font_size=8)

    ax.set_title(title, fontweight='bold', pad=10)
    ax.axis('off')


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
draw_graph(axes[0], 'Full Graph',                          VERTICES, EDGES)
draw_graph(axes[1], f"Kruskal's MST  (cost={k_cost})",    VERTICES, EDGES, k_mst)
draw_graph(axes[2], f"Prim's MST     (cost={p_cost})",    VERTICES, EDGES, p_mst)

red_patch  = mpatches.Patch(color='#e74c3c', label='MST Edge')
gray_patch = mpatches.Patch(color='#bdc3c7', label='Excluded Edge')
fig.legend(handles=[red_patch, gray_patch], loc='lower center', ncol=2, frameon=False)

plt.suptitle('Minimum Spanning Tree Comparison', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('mst_visualization.png', bbox_inches='tight')
plt.show()

print(f"Kruskal's cost : {k_cost}")
print(f"Prim's cost    : {p_cost}")
print(f'Both agree     : {k_cost == p_cost}')

In [ ]:
# =============================================================================
# RANDOM LARGE GRAPH STRESS TEST
# Generates a random connected graph and verifies both MST algorithms agree.
# =============================================================================

def random_graph(n_vertices=15, target_edges=35, weight_range=(1, 50)):
    """Generate a random connected weighted undirected graph.

    Connectivity is guaranteed by first building a spanning chain (1-2-3-...-n).
    Extra random edges are added until target_edges is reached.

    Edges are keyed by frozenset({u, v}) to prevent parallel edges between the
    same pair of vertices, and stored with canonical ordering (min, max) to keep
    the edge list consistent for both Kruskal's and Prim's.

    Args:
        n_vertices:   Number of vertices (labelled 1..n_vertices).
        target_edges: Desired number of unique edges.
        weight_range: (min, max) weight for edges.

    Returns:
        (vertices_list, edges_list)
    """
    vertices = list(range(1, n_vertices + 1))
    edge_dict = {}   # frozenset({u,v}) -> (u, v, w)

    # Spanning chain guarantees connectivity
    for i in range(1, n_vertices):
        w   = random.randint(*weight_range)
        key = frozenset({i, i + 1})
        edge_dict[key] = (i, i + 1, w)

    # Fill to target_edges with additional random edges
    max_attempts = target_edges * 20
    attempts = 0
    while len(edge_dict) < target_edges and attempts < max_attempts:
        u = random.randint(1, n_vertices)
        v = random.randint(1, n_vertices)
        attempts += 1
        if u == v:
            continue
        key = frozenset({u, v})
        if key not in edge_dict:
            w = random.randint(*weight_range)
            edge_dict[key] = (min(u, v), max(u, v), w)   # canonical ordering

    return vertices, list(edge_dict.values())


rnd_vertices, rnd_edges = random_graph(n_vertices=15, target_edges=40)
rnd_adj = build_adj(rnd_edges)

start_k = time.perf_counter()
km, kc  = kruskal(rnd_vertices, rnd_edges, verbose=False)
time_k  = (time.perf_counter() - start_k) * 1000

start_p = time.perf_counter()
pm, pc  = prim(rnd_vertices, rnd_adj, verbose=False)
time_p  = (time.perf_counter() - start_p) * 1000

print(f'Random graph   : {len(rnd_vertices)} vertices, {len(rnd_edges)} edges')
print(f"Kruskal's MST  : cost={kc}  ({time_k:.4f} ms)")
print(f"Prim's MST     : cost={pc}  ({time_p:.4f} ms)")
print(f'Costs match    : {kc == pc}')

---
<a id='part3'></a>
## Part 3 — Recursive Function Simulation

Each recursive function is implemented with:
- **Call-stack tracing** — indented print statements that mirror the actual call tree.
- **Base-case labelling** — explicit messages when a base case is reached.
- **Return-value tracing** — the computed value is shown as each frame unwinds.

| Function | Recurrence | Time Complexity |
|---|---|---|
| Factorial | T(n) = T(n-1) + O(1) | O(n) |
| Fibonacci (naive) | T(n) = T(n-1) + T(n-2) + O(1) | O(2^n) |
| Fibonacci (memoized) | T(n) = O(1) after cache hit | O(n) |
| Tower of Hanoi | T(n) = 2*T(n-1) + O(1) | O(2^n) |
| Binary Search | T(n) = T(n/2) + O(1) | O(log n) |
| Merge Sort | T(n) = 2*T(n/2) + O(n) | O(n log n) |
| GCD (Euclidean) | T(a,b) = T(b, a mod b) + O(1) | O(log(min(a,b))) |
| Fast Power | T(n) = T(n/2) + O(1) | O(log n) |

In [ ]:
# =============================================================================
# INDENTATION HELPER
# =============================================================================

def make_indent(depth, unit='  '):
    """Return a string of `depth` repetitions of `unit` for call-tree indentation."""
    return unit * depth


# =============================================================================
# FACTORIAL
# T(n) = T(n-1) + O(1)  ->  O(n) time, O(n) stack space.
# =============================================================================

def factorial(n, depth=0):
    """Compute n! recursively and print the call tree.

    Args:
        n:     Non-negative integer.
        depth: Recursion depth for indentation (callers always pass 0).

    Returns:
        n!
    """
    pad = make_indent(depth)
    print(f'{pad}factorial({n})')
    if n <= 1:
        print(f'{pad}-> base case: return 1')
        return 1
    result = n * factorial(n - 1, depth + 1)
    print(f'{pad}-> {n} x factorial({n - 1}) = {result}')
    return result


print('=== FACTORIAL(6) ===')
ans = factorial(6)
print(f'\nResult = {ans}  (expected {6*5*4*3*2*1})')

In [ ]:
# =============================================================================
# FIBONACCI — Naive vs. Memoized
# Naive:    O(2^n) time — recomputes the same subproblems exponentially.
# Memoized: O(n) time  — each subproblem is computed exactly once.
# =============================================================================

_fib_cache = {}   # module-level cache for the memoized variant


def fibonacci(n, depth=0, memo=False):
    """Compute the n-th Fibonacci number recursively and print the call tree.

    fib(0) = 0, fib(1) = 1, fib(n) = fib(n-1) + fib(n-2) for n >= 2.

    Args:
        n:     Non-negative integer index.
        depth: Recursion depth for indentation (callers always pass 0).
        memo:  If True, use _fib_cache to avoid recomputation.

    Returns:
        The n-th Fibonacci number.
    """
    pad = make_indent(depth)

    if memo and n in _fib_cache:
        print(f'{pad}fibonacci({n}) -> cache hit: {_fib_cache[n]}')
        return _fib_cache[n]

    print(f'{pad}fibonacci({n})')
    if n <= 1:
        print(f'{pad}-> base case: return {n}')
        return n

    left_val  = fibonacci(n - 1, depth + 1, memo)
    right_val = fibonacci(n - 2, depth + 1, memo)
    result    = left_val + right_val

    if memo:
        _fib_cache[n] = result

    print(f'{pad}-> fib({n-1}) + fib({n-2}) = {left_val} + {right_val} = {result}')
    return result


print('=== FIBONACCI(6) — Naive Recursive (O(2^n)) ===')
ans = fibonacci(6, memo=False)
print(f'\nResult = {ans}  (expected 8)')

In [ ]:
# Memoized run — clear the cache first so this cell is idempotent across re-runs
_fib_cache.clear()
print('=== FIBONACCI(8) — Memoized (O(n)) ===')
ans = fibonacci(8, memo=True)
print(f'\nResult = {ans}  (expected 21)')

In [ ]:
# =============================================================================
# TOWER OF HANOI
# T(n) = 2*T(n-1) + O(1)  ->  O(2^n) time, exactly 2^n - 1 moves.
# The function returns its moves as a list so re-running the cell always
# produces the correct count (no module-level accumulation).
# =============================================================================

def tower_of_hanoi(n, source='A', auxiliary='B', destination='C', depth=0):
    """Solve Tower of Hanoi for n disks and print every move.

    Args:
        n:           Number of disks.
        source:      Peg to move disks from.
        auxiliary:   Intermediate peg.
        destination: Target peg.
        depth:       Recursion depth for indentation.

    Returns:
        List of move strings describing the complete solution.
    """
    pad = make_indent(depth)
    print(f'{pad}hanoi({n}, {source} -> {destination})')

    if n == 1:
        move = f'Move disk 1 from {source} to {destination}'
        print(f'{pad}  -> {move}')
        return [move]

    moves = []
    moves.extend(tower_of_hanoi(n - 1, source, destination, auxiliary, depth + 1))
    move = f'Move disk {n} from {source} to {destination}'
    print(f'{pad}  -> {move}')
    moves.append(move)
    moves.extend(tower_of_hanoi(n - 1, auxiliary, source, destination, depth + 1))
    return moves


print('=== TOWER OF HANOI (4 disks) ===')
hanoi_solution = tower_of_hanoi(4)
expected_moves = 2**4 - 1
print(f'\nTotal moves: {len(hanoi_solution)}  (expected 2^4 - 1 = {expected_moves})')
assert len(hanoi_solution) == expected_moves, 'Move count mismatch!'

In [ ]:
# =============================================================================
# BONUS — BINARY SEARCH (recursive)
# T(n) = T(n/2) + O(1)  ->  O(log n) time.
# Requires the input array to be sorted.
# =============================================================================

def binary_search(arr, target, lo=0, hi=None, depth=0):
    """Recursively search for target in sorted arr[lo..hi].

    Args:
        arr:    Sorted list of comparable elements.
        target: Value to search for.
        lo:     Left boundary (inclusive). Defaults to 0.
        hi:     Right boundary (inclusive). Defaults to len(arr) - 1.
        depth:  Recursion depth for indentation.

    Returns:
        Index of target in arr, or -1 if not present.
    """
    if hi is None:
        hi = len(arr) - 1

    pad = make_indent(depth)
    print(f'{pad}binary_search(arr[{lo}..{hi}], target={target})')

    if lo > hi:
        print(f'{pad}-> not found')
        return -1

    mid = (lo + hi) // 2
    print(f'{pad}-> mid={mid}, arr[mid]={arr[mid]}')

    if arr[mid] == target:
        print(f'{pad}-> FOUND at index {mid}')
        return mid
    if arr[mid] < target:
        return binary_search(arr, target, mid + 1, hi, depth + 1)
    return binary_search(arr, target, lo, mid - 1, depth + 1)


sorted_list = list(range(0, 50, 2))   # [0, 2, 4, ..., 48]
print('=== BINARY SEARCH ===')
print(f'Array: {sorted_list}\n')
result_idx = binary_search(sorted_list, target=36)
print(f'\nFound at index: {result_idx}  (expected 18)')

In [ ]:
# =============================================================================
# BONUS — MERGE SORT TRACE (recursive)
# Shows the full divide-and-conquer tree including every merge step.
# =============================================================================

def merge_sort_trace(arr, depth=0):
    """Sort arr with merge sort and print the complete divide/merge call tree.

    Args:
        arr:   List to sort (input is not modified).
        depth: Recursion depth for indentation.

    Returns:
        Sorted list.
    """
    pad = make_indent(depth)
    print(f'{pad}merge_sort({arr})')

    if len(arr) <= 1:
        print(f'{pad}-> base case: {arr}')
        return arr

    mid          = len(arr) // 2
    left_sorted  = merge_sort_trace(arr[:mid], depth + 1)
    right_sorted = merge_sort_trace(arr[mid:], depth + 1)

    # Merge the two sorted halves
    merged = []
    i = j = 0
    while i < len(left_sorted) and j < len(right_sorted):
        if left_sorted[i] <= right_sorted[j]:
            merged.append(left_sorted[i])
            i += 1
        else:
            merged.append(right_sorted[j])
            j += 1
    # Append any remaining elements (at most one half is non-empty here)
    merged.extend(left_sorted[i:])
    merged.extend(right_sorted[j:])

    print(f'{pad}-> merge({left_sorted}, {right_sorted}) = {merged}')
    return merged


print('=== MERGE SORT RECURSIVE TRACE ===')
sorted_result = merge_sort_trace([5, 2, 8, 1, 9, 3])
print(f'\nFinal sorted: {sorted_result}')

In [ ]:
# =============================================================================
# BONUS — GREATEST COMMON DIVISOR (Euclidean algorithm)
# T(a,b) = T(b, a mod b) + O(1)  ->  O(log(min(a,b))) time.
# =============================================================================

def gcd(a, b, depth=0):
    """Compute gcd(a, b) via the Euclidean algorithm and print the call tree.

    Base case: gcd(a, 0) = a.
    Recursive case: gcd(a, b) = gcd(b, a mod b).

    Args:
        a, b:  Non-negative integers.
        depth: Recursion depth for indentation.

    Returns:
        Greatest common divisor of a and b.
    """
    pad = make_indent(depth)
    print(f'{pad}gcd({a}, {b})')
    if b == 0:
        print(f'{pad}-> base case: return {a}')
        return a
    result = gcd(b, a % b, depth + 1)
    print(f'{pad}-> gcd({b}, {a} mod {b} = {a % b}) = {result}')
    return result


print('=== GCD(48, 18) ===')
ans = gcd(48, 18)
print(f'\nResult = {ans}  (expected 6)')

In [ ]:
# =============================================================================
# BONUS — FAST EXPONENTIATION (exponentiation by squaring)
# T(n) = T(n/2) + O(1)  ->  O(log n) time.
# Key identity: base^n = (base^(n/2))^2  for even n.
# =============================================================================

def power(base, exp, depth=0):
    """Compute base^exp using fast exponentiation (exponentiation by squaring).

    Args:
        base:  The base value.
        exp:   Non-negative integer exponent.
        depth: Recursion depth for indentation.

    Returns:
        base raised to the power exp.
    """
    pad = make_indent(depth)
    print(f'{pad}power({base}, {exp})')

    if exp == 0:
        print(f'{pad}-> base case: return 1')
        return 1

    if exp % 2 == 0:
        half   = power(base, exp // 2, depth + 1)
        result = half * half
        print(f'{pad}-> (base^{exp//2})^2 = {half}^2 = {result}')
    else:
        result = base * power(base, exp - 1, depth + 1)
        print(f'{pad}-> {base} x power({base}, {exp - 1}) = {result}')

    return result


print('=== FAST POWER: 2^10 ===')
ans = power(2, 10)
print(f'\nResult = {ans}  (expected {2**10})')

In [ ]:
# =============================================================================
# RECURSION DEPTH & CALL COUNT PROFILER
# Wraps any recursive function transparently to track max recursion depth
# and total number of recursive calls.
# =============================================================================

class RecursionProfiler:
    """Transparent wrapper that counts calls and tracks max recursion depth.

    How it works:
        Redirect the global name used by the function to a profiler instance.
        Because Python looks up global names at call time, subsequent recursive
        calls to the original global name will all pass through the profiler.

    Example::
        def my_fn(n): return 1 if n <= 1 else my_fn(n - 1)
        profiler = RecursionProfiler(my_fn)
        my_fn    = profiler          # redirect global name
        my_fn(5)
        print(profiler.calls, profiler.max_depth)
        profiler.reset()
    """

    def __init__(self, fn):
        self.fn          = fn
        self.calls       = 0
        self.max_depth   = 0
        self._cur_depth  = 0

    def __call__(self, *args, **kwargs):
        self.calls      += 1
        self._cur_depth += 1
        self.max_depth   = max(self.max_depth, self._cur_depth)
        result = self.fn(*args, **kwargs)
        self._cur_depth -= 1
        return result

    def reset(self):
        """Reset all counters to zero."""
        self.calls = self.max_depth = self._cur_depth = 0


# Non-tracing standalone functions for clean profiling
def _factorial_plain(n): return 1 if n <= 1 else n * _factorial_plain(n - 1)
def _fibonacci_plain(n): return n if n <= 1 else _fibonacci_plain(n - 1) + _fibonacci_plain(n - 2)

# Redirect global names through profilers so all recursive calls are counted
profiled_factorial = RecursionProfiler(_factorial_plain)
_factorial_plain   = profiled_factorial

profiled_fibonacci = RecursionProfiler(_fibonacci_plain)
_fibonacci_plain   = profiled_fibonacci

header = f'{"n":>4}  {"fact calls":>10}  {"fact depth":>10}  {"fib calls":>10}  {"fib depth":>10}'
print(header)
print('-' * len(header))
for n in range(1, 13):
    profiled_factorial.reset()
    profiled_fibonacci.reset()
    _factorial_plain(n)
    _fibonacci_plain(n)
    print(f'{n:>4}  {profiled_factorial.calls:>10}  {profiled_factorial.max_depth:>10}'
          f'  {profiled_fibonacci.calls:>10}  {profiled_fibonacci.max_depth:>10}')

In [ ]:
# =============================================================================
# RECURSION CALL GROWTH CHART
# Visualises O(n) factorial growth vs. O(2^n) naive Fibonacci growth.
# Fresh functions are defined here to avoid depending on the reassignments
# made in the profiler cell above.
# =============================================================================

def _fact_count(n): return 1 if n <= 1 else n * _fact_count(n - 1)
def _fib_count(n):  return n if n <= 1 else _fib_count(n - 1) + _fib_count(n - 2)

ns               = list(range(1, 16))
fact_call_counts = []
fib_call_counts  = []

for n in ns:
    # Profile factorial for this n
    fact_profiler = RecursionProfiler(_fact_count)
    _fact_count   = fact_profiler          # redirect recursive calls through profiler
    fact_profiler(n)
    fact_call_counts.append(fact_profiler.calls)
    _fact_count = fact_profiler.fn         # restore original for next iteration

    # Profile fibonacci for this n
    fib_profiler = RecursionProfiler(_fib_count)
    _fib_count   = fib_profiler
    fib_profiler(n)
    fib_call_counts.append(fib_profiler.calls)
    _fib_count = fib_profiler.fn

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ns, fact_call_counts, 'o-', label='Factorial  O(n)',   color='#3498db', linewidth=2)
ax.plot(ns, fib_call_counts,  's-', label='Fibonacci  O(2^n)', color='#e74c3c', linewidth=2)
ax.set_xlabel('n', fontsize=11)
ax.set_ylabel('Total Recursive Calls', fontsize=11)
ax.set_title('Recursive Call Growth: Factorial O(n) vs. Fibonacci O(2^n)', fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('recursion_call_growth.png', bbox_inches='tight')
plt.show()
print('Chart saved to recursion_call_growth.png')

In [ ]:
# =============================================================================
# TOWER OF HANOI — Move Count vs. Number of Disks
# Demonstrates 2^n - 1 exponential growth visually.
# An assertion verifies the simulation matches the closed-form formula.
# =============================================================================

def count_hanoi_moves(n):
    """Return the number of disk moves required to solve Hanoi for n disks.

    Uses a silent simulation (no print output) for efficiency.
    The expected answer is 2^n - 1.

    Args:
        n: Number of disks (positive integer).

    Returns:
        Integer move count.
    """
    move_count = [0]

    def _hanoi(disks, src, aux, dst):
        if disks == 1:
            move_count[0] += 1
            return
        _hanoi(disks - 1, src, dst, aux)
        move_count[0] += 1
        _hanoi(disks - 1, aux, src, dst)

    _hanoi(n, 'A', 'B', 'C')
    return move_count[0]


disk_range   = list(range(1, 17))
actual_moves = [count_hanoi_moves(n) for n in disk_range]
theoretical  = [2**n - 1 for n in disk_range]

assert actual_moves == theoretical, 'Simulation does not match 2^n - 1 formula!'

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(disk_range, actual_moves, color='#9b59b6', alpha=0.8, label='Actual moves (simulation)')
ax.plot(disk_range, theoretical, 'r--o', label='2^n - 1 (theoretical)', linewidth=1.5)
ax.set_xlabel('Number of Disks', fontsize=11)
ax.set_ylabel('Total Moves', fontsize=11)
ax.set_title('Tower of Hanoi — Exponential Move Growth (2^n - 1)', fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('hanoi_growth.png', bbox_inches='tight')
plt.show()
print('Chart saved to hanoi_growth.png')

---
<a id='summary'></a>
## Summary

| Component | What was implemented |
|---|---|
| **Part 1 — Sorting** | All 8 algorithms: Bubble, Selection, Insertion, Merge, Quick, Randomized Quick, Counting, Radix |
| | Metrics: wall-clock time (ms, averaged over 5 trials), operation counts (averaged), swap counts (averaged) |
| | Charts: benchmark bar chart, scalability line chart (averaged over 3 trials per size), best/avg/worst pivot table |
| | Step-by-step Bubble Sort trace |
| **Part 2 — MST** | Kruskal's with Union-Find (path compression + union by rank), O(E log E) |
| | Prim's with min-heap, O(E log V) |
| | Step-by-step trace for both algorithms on a sample 6-vertex graph |
| | Side-by-side NetworkX visualization |
| | Random large-graph stress test with parallel-edge-safe generation |
| **Part 3 — Recursion** | Factorial, Fibonacci (naive + memoized), Tower of Hanoi |
| | Bonus: Binary Search, Merge Sort trace, GCD (Euclidean), Fast Exponentiation |
| | Transparent RecursionProfiler measuring total call count and max recursion depth |
| | Charts: O(n) vs O(2^n) call growth; Hanoi exponential growth with formula verification |